STEP 1 — Import libraries and load processed dataset

In [2]:

# STEP 1: IMPORT LIBRARIES AND LOAD DATASET

import pandas as pd
import numpy as np
import os
import json

from pathlib import Path

# Dataset path
dataset_path = Path(
    "processed_dataset_1/dataset1_pe_features.csv"
)

# Verify file
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path.resolve()}"
    )

# Load dataset
df_ml = pd.read_csv(dataset_path)

print("=" * 60)
print("DATASET 1 - ML PREPARATION")
print("=" * 60)

print("\nDATASET SHAPE:", df_ml.shape)

print("\nFIRST 5 RECORDS:")
display(df_ml.head())

print("\nDATA TYPES:")
print(df_ml.dtypes.value_counts())

print("\nTOTAL MISSING VALUES:")
print(df_ml.isna().sum().sum())


DATASET 1 - ML PREPARATION

DATASET SHAPE: (9998, 652)

FIRST 5 RECORDS:


,header_coff_timestamp,header_coff_number_of_sections,header_coff_number_of_symbols,header_coff_sizeof_optional_header,header_coff_pointer_to_symbol_table,header_coff_characteristics_count,header_coff_characteristics_LARGE_ADDRESS_AWARE,header_optional_major_image_version,header_optional_minor_image_version,header_optional_major_linker_version,...,section_entry_.upx1,section_entry_.vixen1,section_entry_.vmp1,section_entry_.vmp2,section_entry_.zdata,section_entry___OTHER__,section_entry_eintim,section_entry_init,section_entry_upx0,section_entry_upx1
0,1.726320e+09,6.0,0.0,240.0,0.0,2.0,1.0,0.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0
1,1.724938e+09,6.0,0.0,240.0,0.0,2.0,1.0,0.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0
2,1.699507e+09,5.0,0.0,240.0,0.0,3.0,1.0,6.0,1.0,9.0,...,0,0,0,0,0,0,0,0,0,0
3,1.710213e+09,6.0,0.0,240.0,0.0,3.0,1.0,0.0,0.0,12.0,...,0,0,0,0,0,0,0,0,0,0
4,1.590101e+09,6.0,0.0,240.0,0.0,3.0,1.0,10.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0



DATA TYPES:
float64    605
int64       47
Name: count, dtype: int64

TOTAL MISSING VALUES:
0


STEP 2 — Verify target label
This is the most important check before training.

In [3]:

# STEP 2: TARGET LABEL VERIFICATION

print("TARGET LABEL INVESTIGATION")
print("=" * 60)

# Find possible target columns
target_candidates = [
    col for col in df_ml.columns
    if col.lower() in [
        "label",
        "target",
        "class",
        "malware_label",
        "is_malware"
    ]
]

print("\nPOSSIBLE TARGET COLUMNS:")
print(target_candidates)

if "label" in df_ml.columns:
    
    print("\nLABEL DATA TYPE:")
    print(df_ml["label"].dtype)

    print("\nLABEL DISTRIBUTION:")
    print(df_ml["label"].value_counts(dropna=False))

    print("\nLABEL DISTRIBUTION (%):")
    print(
        df_ml["label"].value_counts(
            normalize=True,
            dropna=False
        ) * 100
    )

    print("\nUNIQUE LABEL VALUES:")
    print(df_ml["label"].unique())

else:
    print("\nWARNING: LABEL COLUMN NOT FOUND.")
    print("Do not proceed to model training.")


TARGET LABEL INVESTIGATION

POSSIBLE TARGET COLUMNS:
[]

Do not proceed to model training.


STEP 3 — Identify identifiers and leakage-prone columns
We must prevent the model from learning sample identities instead of malware characteristics.

In [4]:

# STEP 3: IDENTIFIER AND DATA LEAKAGE INSPECTION

leakage_keywords = [
    "sha256",
    "sha1",
    "md5",
    "tlsh",
    "hash",
    "filename",
    "file_name",
    "sample_id",
    "malware_family",
    "family",
    "av_verdict",
    "antivirus",
    "detection_name",
    "first_seen",
    "source",
    "tags"
]

identifier_columns = [
    col for col in df_ml.columns
    if any(
        keyword in col.lower()
        for keyword in leakage_keywords
    )
]

print("IDENTIFIER / POTENTIAL LEAKAGE COLUMNS:")
print(identifier_columns)

print("\nTOTAL IDENTIFIED:")
print(len(identifier_columns))

print("\nTARGET COLUMN PRESENT:", "label" in df_ml.columns)


IDENTIFIER / POTENTIAL LEAKAGE COLUMNS:
[]

TOTAL IDENTIFIED:
0

TARGET COLUMN PRESENT: False


STEP 4 — Check duplicate records and feature integrity

In [5]:

# STEP 4: DUPLICATE AND FEATURE INTEGRITY CHECK

print("FEATURE INTEGRITY REPORT")
print("=" * 60)

print("Total records:", len(df_ml))

# Duplicate complete rows
print(
    "Duplicate complete rows:",
    df_ml.duplicated().sum()
)

# Check duplicate SHA256 if available
if "sha256" in df_ml.columns:
    print(
        "Duplicate SHA256:",
        df_ml["sha256"].duplicated().sum()
    )

# Check constant columns
constant_columns = [
    col for col in df_ml.columns
    if df_ml[col].nunique(dropna=False) <= 1
]

print("\nConstant columns:", len(constant_columns))
print(constant_columns)

# Check infinite numerical values
numeric_df = df_ml.select_dtypes(include=np.number)

print(
    "\nInfinite values:",
    np.isinf(numeric_df).sum().sum()
)

print("\nIntegrity inspection completed.")


FEATURE INTEGRITY REPORT
Total records: 9998
Duplicate complete rows: 0

Constant columns: 0
[]

Infinite values: 0

Integrity inspection completed.


STEP 5 — Initial class distribution visualization

In [6]:

# STEP 5: CLASS DISTRIBUTION

import matplotlib.pyplot as plt

if "label" in df_ml.columns:

    label_counts = df_ml["label"].value_counts().sort_index()

    print(label_counts)

    label_counts.plot(
        kind="bar",
        figsize=(7, 4)
    )

    plt.title("Dataset 1 - Target Class Distribution")
    plt.xlabel("Label")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

else:
    print("Target label unavailable. Skipping visualization.")


Matplotlib is building the font cache; this may take a moment.


Target label unavailable. Skipping visualization.


STEP 6 — Investigate original dataset
Run this cell in Notebook 04. It searches your project directory for CSV files and checks their column names without loading entire datasets.

In [7]:

# STEP 6: FIND ORIGINAL TARGET LABEL

from pathlib import Path
import pandas as pd

project_dir = Path.cwd()

print("PROJECT DIRECTORY:", project_dir)

csv_files = list(project_dir.rglob("*.csv"))

print("\nCSV FILES FOUND:", len(csv_files))

for file in csv_files:

    try:
        temp = pd.read_csv(file, nrows=5)

        possible_labels = [
            col for col in temp.columns
            if any(word in col.lower() for word in [
                "label", "class", "target",
                "malware", "benign", "verdict"
            ])
        ]

        print("\nFILE:", file.name)
        print("COLUMNS:", len(temp.columns))
        print("POSSIBLE LABELS:", possible_labels)

    except Exception as e:
        print("\nSKIPPED:", file.name, str(e)[:150])


PROJECT DIRECTORY: /Users/gouravsingh/Desktop/Ai-Ml-integrated-SOC-Platform/notebook

CSV FILES FOUND: 2

FILE: removed_duplicate_features.csv
COLUMNS: 2
POSSIBLE LABELS: []

FILE: dataset1_pe_features.csv
COLUMNS: 652
POSSIBLE LABELS: ['section_name_.targetc_count']


STEP 7 — Inspect original raw CSV specifically
Your VS Code screenshot shows dataset1_raw_20000.csv. Let's inspect its schema and sample records.

In [12]:

# STEP 8: INSPECT ORIGINAL RAW DATASET

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent

raw_path = (
    project_root /
    "data" /
    "processed" /
    "dataset_1_raw_20000.csv"
)

print("RAW FILE:", raw_path)
print("FILE EXISTS:", raw_path.exists())

if raw_path.exists():

    df_raw_check = pd.read_csv(raw_path, nrows=5)

    print("\nCOLUMN COUNT:", len(df_raw_check.columns))
    print("\nALL COLUMNS:")
    print(df_raw_check.columns.tolist())

    possible_labels = [
        col for col in df_raw_check.columns
        if any(word in col.lower() for word in [
            "label", "class", "target",
            "malware", "benign", "verdict"
        ])
    ]

    print("\nPOSSIBLE LABEL COLUMNS:", possible_labels)

    print("\nFIRST 5 RECORDS:")
    display(df_raw_check)

else:
    print("ERROR: Raw dataset not found.")


RAW FILE: /Users/gouravsingh/Desktop/Ai-Ml-integrated-SOC-Platform/data/processed/dataset_1_raw_20000.csv
FILE EXISTS: True

COLUMN COUNT: 32

ALL COLUMNS:
['md5', 'sha1', 'sha256', 'tlsh', 'first_submission_date', 'last_analysis_date', 'detection_ratio', 'label', 'file_type', 'family', 'family_confidence', 'behavior', 'file_property', 'packer', 'exploit', 'group', 'histogram', 'byteentropy', 'strings', 'general', 'header', 'section', 'imports', 'exports', 'datadirectories', 'richheader', 'authenticode', 'pefilewarnings', 'week_id', 'caps', 'ttps', 'mbc']

POSSIBLE LABEL COLUMNS: ['label']

FIRST 5 RECORDS:


,md5,sha1,sha256,tlsh,first_submission_date,last_analysis_date,detection_ratio,label,file_type,family,...,imports,exports,datadirectories,richheader,authenticode,pefilewarnings,week_id,caps,ttps,mbc
0,9c6701567cbcf50e11810c0a1ca2a580,43e53f8e44ecd49b55c8f5ed9ae33b8dd8e9534d,00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...,T1B9C6338D77150C9DE95FA239E4E1863766B375B50BA0...,1726320821,1730346557,12/73,1,Win64,NaN,...,"{'USER32.dll': ['CreateWindowExW', 'ShutdownBl...",[],"[{'has_relocs': 1, 'has_dynamic_relocs': 0}, {...","[17004619, 7, 17135691, 180, 17070155, 10, 166...","{'num_certs': 0, 'self_signed': 0, 'empty_prog...",['Failed parsing FunctionEntry of UNWIND_INFO ...,50,[],[],[]
1,600de330e2b039242fe26276abb5ab47,237e975cdc1b33f385779e126bf2905b15188f21,0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9e...,T1F8F45B4B626802FCD5ABD17886579E62E776B40A0361...,1725974713,1728999110,0/74,0,Win64,NaN,...,"{'dwmapi.dll': ['DwmSetWindowAttribute'], 'app...",['?FirebaseCorePluginCApiRegisterWithRegistrar...,"[{'has_relocs': 1, 'has_dynamic_relocs': 0}, {...","[9664521, 22, 17072578, 10, 17007042, 3, 17138...","{'num_certs': 1, 'self_signed': 0, 'empty_prog...",['Failed parsing FunctionEntry of UNWIND_INFO ...,50,"[{'Capability': 'Find graphical window', 'Name...","[{'Tactic': 'DISCOVERY', 'Technique': 'Applica...","[{'Objective': 'PROCESS', 'Behavior': 'Create ..."
2,900e6647f6045831ffa57f9bb34c5da3,10197906ecf11324238bc9b39637e23919e4e4c3,001aea7ddef769f8b44ffc497cdf4f05193b4889510a53...,T1D1844A4666A5C760E02182F1DBB6CAAF793EFC011B24...,1726316956,1729341470,0/74,0,Win64,NaN,...,"{'msvcrt.dll': ['free', '_wtoi', 'calloc', '_i...","['DllCanUnloadNow', 'DllGetClassObject']","[{'has_relocs': 1, 'has_dynamic_relocs': 0}, {...","[9795593, 2, 65536, 93, 9664521, 15, 8615945, ...","{'num_certs': 1, 'self_signed': 1, 'empty_prog...",['Byte 0x...'],50,"[{'Capability': 'Print debug messages', 'Names...",[],"[{'Objective': 'ANTI-BEHAVIORAL ANALYSIS', 'Be..."
3,6f87d8b7205b291fb3437a5d6c54864d,b791844b2e9346ba2d4637b8c8e4f2777c5fb136,001b30621be4731fbd66ce0480ffcf15f8feeebedd9b15...,T16C035B9F27191445F6B7433CC5C6CE13EE7076612B63...,1725931065,1733202100,0/73,0,Win64,NaN,...,"{'SETUPAPI.dll': ['SetupDiGetClassDevsW', 'Set...","['MCCopyDevNodeBusDescription', 'MCCopyDevNode...","[{'has_relocs': 1, 'has_dynamic_relocs': 0}, {...","[14635334, 1, 14700870, 12, 14766406, 2, 14504...","{'num_certs': 1, 'self_signed': 0, 'empty_prog...",[],50,[{'Capability': 'Link function at runtime on w...,"[{'Tactic': 'EXECUTION', 'Technique': 'Shared ...",[]
4,803dfd57fa9d75a4520cef396eeb4de6,d8a273e10a5d6ed97fddefd59df518405a93fcbb,001dc027ca8aea62fdf19327d2a74ddf156aef93c8cbb0...,T10A044A46B95D02BEC4A3B4788479312AF53234145BB1...,1726251086,1731646884,0/74,0,Win64,NaN,...,{'MSVCP140.dll': ['?widen@?$ctype@D@std@@QEBAP...,"['__std_smf_assoc_laguerre', '__std_smf_assoc_...","[{'has_relocs': 1, 'has_dynamic_relocs': 0}, {...","[15703220, 2, 9664521, 12, 17067979, 8, 170024...","{'num_certs': 1, 'self_signed': 0, 'empty_prog...",[],50,"[{'Capability': 'Encode data using xor', 'Name...","[{'Tactic': 'DEFENSE EVASION', 'Technique': 'O...","[{'Objective': 'DEFENSE EVASION', 'Behavior': ..."


STEP 9 — Inspect ground-truth labels

In [13]:

# STEP 9: GROUND-TRUTH LABEL ANALYSIS

raw_path = (
    project_root / "data" / "processed" /
    "dataset_1_raw_20000.csv"
)

df_original = pd.read_csv(raw_path)

print("=" * 60)
print("ORIGINAL DATASET LABEL ANALYSIS")
print("=" * 60)

print("\nDATASET SHAPE:", df_original.shape)

print("\nLABEL DATA TYPE:")
print(df_original["label"].dtype)

print("\nUNIQUE LABEL VALUES:")
print(df_original["label"].unique())

print("\nLABEL DISTRIBUTION:")
print(df_original["label"].value_counts(dropna=False))

print("\nLABEL DISTRIBUTION (%):")
print(
    df_original["label"]
    .value_counts(normalize=True, dropna=False) * 100
)

print("\nMISSING LABELS:")
print(df_original["label"].isna().sum())

print("\nSHA256 AVAILABILITY:")
print("Column exists:", "sha256" in df_original.columns)
print("Missing SHA256:", df_original["sha256"].isna().sum())
print("Duplicate SHA256:", df_original["sha256"].duplicated().sum())

print("\nPROCESSED DATASET IDENTIFIER CHECK:")
print("SHA256 in df_ml:", "sha256" in df_ml.columns)


ORIGINAL DATASET LABEL ANALYSIS

DATASET SHAPE: (20000, 32)

LABEL DATA TYPE:
int64

UNIQUE LABEL VALUES:
[1 0]

LABEL DISTRIBUTION:
label
1    10000
0    10000
Name: count, dtype: int64

LABEL DISTRIBUTION (%):
label
1    50.0
0    50.0
Name: proportion, dtype: float64

MISSING LABELS:
0

SHA256 AVAILABILITY:
Column exists: True
Missing SHA256: 0
Duplicate SHA256: 10000

PROCESSED DATASET IDENTIFIER CHECK:
SHA256 in df_ml: False


STEP 10 — Investigate duplicate hashes and consolidated dataset

In [14]:

# STEP 10: HASH INTEGRITY AND LABEL CONSISTENCY

# 1. Check SHA256 groups
hash_analysis = (
    df_original.groupby("sha256")
    .agg(
        record_count=("label", "size"),
        unique_labels=("label", "nunique"),
        labels=("label", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

print("UNIQUE SHA256:", len(hash_analysis))

print("\nHASH FREQUENCY:")
print(hash_analysis["record_count"].value_counts())

print("\nHASHES WITH CONFLICTING LABELS:")
conflicts = hash_analysis[
    hash_analysis["unique_labels"] > 1
]
print(len(conflicts))

print("\nHASHES WITH REPEATED IDENTICAL LABELS:")
print(
    len(hash_analysis[
        hash_analysis["record_count"] > 1
    ]) - len(conflicts)
)

# 2. Inspect consolidated metadata
consolidated_path = (
    project_root / "data" / "processed" /
    "dataset_1_consolidated_10000.csv"
)

df_consolidated = pd.read_csv(consolidated_path)

print("\nCONSOLIDATED DATASET SHAPE:")
print(df_consolidated.shape)

print("\nCONSOLIDATED COLUMNS:")
print(df_consolidated.columns.tolist())

print("\nCONSOLIDATED LABEL DISTRIBUTION:")
if "label" in df_consolidated.columns:
    print(df_consolidated["label"].value_counts(dropna=False))

# 3. Inspect feature dataset
feature_path = (
    project_root / "notebook" /
    "processed_dataset_1" /
    "dataset1_pe_features.csv"
)

print("\nFEATURE FILE EXISTS:", feature_path.exists())

if feature_path.exists():
    df_features_check = pd.read_csv(
        feature_path, nrows=5
    )

    print("FEATURE COLUMNS:", len(df_features_check.columns))
    print("FEATURE SHAPE SAMPLE:")
    display(df_features_check.head())


UNIQUE SHA256: 10000

HASH FREQUENCY:
record_count
2    10000
Name: count, dtype: int64

HASHES WITH CONFLICTING LABELS:
0

HASHES WITH REPEATED IDENTICAL LABELS:
10000

CONSOLIDATED DATASET SHAPE:
(10000, 32)

CONSOLIDATED COLUMNS:
['md5', 'sha1', 'sha256', 'tlsh', 'first_submission_date', 'last_analysis_date', 'detection_ratio', 'label', 'file_type', 'family', 'family_confidence', 'behavior', 'file_property', 'packer', 'exploit', 'group', 'histogram', 'byteentropy', 'strings', 'general', 'header', 'section', 'imports', 'exports', 'datadirectories', 'richheader', 'authenticode', 'pefilewarnings', 'week_id', 'caps', 'ttps', 'mbc']

CONSOLIDATED LABEL DISTRIBUTION:
label
1    5000
0    5000
Name: count, dtype: int64

FEATURE FILE EXISTS: True
FEATURE COLUMNS: 652
FEATURE SHAPE SAMPLE:


,header_coff_timestamp,header_coff_number_of_sections,header_coff_number_of_symbols,header_coff_sizeof_optional_header,header_coff_pointer_to_symbol_table,header_coff_characteristics_count,header_coff_characteristics_LARGE_ADDRESS_AWARE,header_optional_major_image_version,header_optional_minor_image_version,header_optional_major_linker_version,...,section_entry_.upx1,section_entry_.vixen1,section_entry_.vmp1,section_entry_.vmp2,section_entry_.zdata,section_entry___OTHER__,section_entry_eintim,section_entry_init,section_entry_upx0,section_entry_upx1
0,1.726320e+09,6.0,0.0,240.0,0.0,2.0,1.0,0.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0
1,1.724938e+09,6.0,0.0,240.0,0.0,2.0,1.0,0.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0
2,1.699507e+09,5.0,0.0,240.0,0.0,3.0,1.0,6.0,1.0,9.0,...,0,0,0,0,0,0,0,0,0,0
3,1.710213e+09,6.0,0.0,240.0,0.0,3.0,1.0,0.0,0.0,12.0,...,0,0,0,0,0,0,0,0,0,0
4,1.590101e+09,6.0,0.0,240.0,0.0,3.0,1.0,10.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0


STEP 11 — Investigate feature-to-label mapping

In [15]:

# STEP 11: FEATURE AND LABEL MAPPING INVESTIGATION

print("CONSOLIDATED DATASET:")
print(df_consolidated.shape)
print(df_consolidated.columns.tolist())

print("\nFEATURE DATASET:")
print(df_features_check.columns.tolist())

# Check available exception files
print("\nPROCESSED DATASET FILES:")

for folder in [
    project_root / "notebook" / "processed_dataset_1",
    project_root / "data" / "processed"
]:
    print(f"\n{folder}:")
    if folder.exists():
        for file in folder.iterdir():
            print(file.name)

# Check common columns
common = set(df_consolidated.columns) & set(
    df_features_check.columns
)

print("\nCOMMON COLUMNS:", common)

# Check whether feature rows retain an original index
print("\nFEATURE INDEX SAMPLE:")
print(df_features_check.index[:10].tolist())

print("\nCONSOLIDATED HASH SAMPLE:")
print(df_consolidated["sha256"].head().tolist())


CONSOLIDATED DATASET:
(10000, 32)
['md5', 'sha1', 'sha256', 'tlsh', 'first_submission_date', 'last_analysis_date', 'detection_ratio', 'label', 'file_type', 'family', 'family_confidence', 'behavior', 'file_property', 'packer', 'exploit', 'group', 'histogram', 'byteentropy', 'strings', 'general', 'header', 'section', 'imports', 'exports', 'datadirectories', 'richheader', 'authenticode', 'pefilewarnings', 'week_id', 'caps', 'ttps', 'mbc']

FEATURE DATASET:
['header_coff_timestamp', 'header_coff_number_of_sections', 'header_coff_number_of_symbols', 'header_coff_sizeof_optional_header', 'header_coff_pointer_to_symbol_table', 'header_coff_characteristics_count', 'header_coff_characteristics_LARGE_ADDRESS_AWARE', 'header_optional_major_image_version', 'header_optional_minor_image_version', 'header_optional_major_linker_version', 'header_optional_minor_linker_version', 'header_optional_major_operating_system_version', 'header_optional_minor_operating_system_version', 'header_optional_major_sub

STEP 12 — Find the original feature-to-file mapping
Run this one cell. It checks whether the original extraction process retained filenames, hashes, or source-file ordering.

In [16]:

# STEP 12: INVESTIGATE ORIGINAL FEATURE SOURCE MAPPING

print("RELEVANT VARIABLES IN MEMORY:")

for name, obj in list(globals().items()):
    if any(word in name.lower() for word in [
        "hash", "file", "source", "record",
        "path", "feature", "exception"
    ]):
        if hasattr(obj, "__len__"):
            try:
                print(name, type(obj).__name__, len(obj))
            except:
                pass

print("\nFEATURE DATAFRAME DETAILS:")
print("Shape:", df_features_check.shape)
print("Index:", df_features_check.index[:10].tolist())

print("\nEXCEPTION REPORT:")
if "exception_report" in globals():
    display(exception_report)
elif "exception_records" in globals():
    display(exception_records)

print("\nSOURCE DATASET SAMPLE:")
display(df_consolidated[
    ["sha256", "label"]
].head(10))


RELEVANT VARIABLES IN MEMORY:
__vsc_ipynb_file__ str 99
csv_files list 2
hash_analysis DataFrame 10000
df_features_check DataFrame 5

FEATURE DATAFRAME DETAILS:
Shape: (5, 652)
Index: [0, 1, 2, 3, 4]

EXCEPTION REPORT:

SOURCE DATASET SAMPLE:


,sha256,label
0,00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...,1
1,0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9e...,0
2,001aea7ddef769f8b44ffc497cdf4f05193b4889510a53...,0
3,001b30621be4731fbd66ce0480ffcf15f8feeebedd9b15...,0
4,001dc027ca8aea62fdf19327d2a74ddf156aef93c8cbb0...,0
5,001e3da23ae36c7ef6af5850f469234b06fc40779ec8f4...,0
6,00208499e4f2a02320ebdd993446ecd310b026e19cfebb...,1
7,00215869134d30dbe58d0c9c3b462ac33590b0552bf71a...,1
8,0022831516cb4d9e122c17cba922efdccfef0ceaaea9d4...,0
9,0022df1d0967d5de37784ae774e7447b6327fa509aeb87...,0


STEP 12A — Inspect the hash mapping and feature extraction order.

In [17]:

# STEP 12A: INSPECT HASH ANALYSIS AND FEATURE ALIGNMENT

print("HASH ANALYSIS SHAPE:", hash_analysis.shape)
print("\nHASH ANALYSIS COLUMNS:")
print(hash_analysis.columns.tolist())

print("\nHASH ANALYSIS FIRST 5 RECORDS:")
display(hash_analysis.head())

print("\nHASH ANALYSIS LAST 5 RECORDS:")
display(hash_analysis.tail())

print("\nFEATURE SAMPLE COLUMNS:")
print(df_features_check.columns.tolist())

print("\nFEATURE SAMPLE:")
display(df_features_check.head())

print("\nFEATURE INDEX:")
print(df_features_check.index.tolist())

print("\nHASH ANALYSIS INDEX:")
print(hash_analysis.index[:10].tolist())


HASH ANALYSIS SHAPE: (10000, 4)

HASH ANALYSIS COLUMNS:
['sha256', 'record_count', 'unique_labels', 'labels']

HASH ANALYSIS FIRST 5 RECORDS:


,sha256,record_count,unique_labels,labels
0,00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...,2,1,[1]
1,0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9e...,2,1,[0]
2,001aea7ddef769f8b44ffc497cdf4f05193b4889510a53...,2,1,[0]
3,001b30621be4731fbd66ce0480ffcf15f8feeebedd9b15...,2,1,[0]
4,001dc027ca8aea62fdf19327d2a74ddf156aef93c8cbb0...,2,1,[0]



HASH ANALYSIS LAST 5 RECORDS:


,sha256,record_count,unique_labels,labels
9995,fff518897a10ce86a3ea90ce7c157047fa3a49ed50799d...,2,1,[1]
9996,fff72c864d9e1abdc4f3664c0c498aebc2d93e0c6664db...,2,1,[0]
9997,fffaa5f9e281f2e37ab419d39dbe3d9077de04b6d666de...,2,1,[1]
9998,fffee248c97020a0ad868cb0b69a50e69f162e5675680f...,2,1,[0]
9999,fffee6c212c3f92297299b6c2a2fdbffb404cfa6218fe0...,2,1,[0]



FEATURE SAMPLE COLUMNS:
['header_coff_timestamp', 'header_coff_number_of_sections', 'header_coff_number_of_symbols', 'header_coff_sizeof_optional_header', 'header_coff_pointer_to_symbol_table', 'header_coff_characteristics_count', 'header_coff_characteristics_LARGE_ADDRESS_AWARE', 'header_optional_major_image_version', 'header_optional_minor_image_version', 'header_optional_major_linker_version', 'header_optional_minor_linker_version', 'header_optional_major_operating_system_version', 'header_optional_minor_operating_system_version', 'header_optional_major_subsystem_version', 'header_optional_minor_subsystem_version', 'header_optional_sizeof_code', 'header_optional_sizeof_headers', 'header_optional_sizeof_image', 'header_optional_sizeof_initialized_data', 'header_optional_sizeof_uninitialized_data', 'header_optional_sizeof_stack_reserve', 'header_optional_sizeof_stack_commit', 'header_optional_sizeof_heap_reserve', 'header_optional_sizeof_heap_commit', 'header_optional_address_of_entr

,header_coff_timestamp,header_coff_number_of_sections,header_coff_number_of_symbols,header_coff_sizeof_optional_header,header_coff_pointer_to_symbol_table,header_coff_characteristics_count,header_coff_characteristics_LARGE_ADDRESS_AWARE,header_optional_major_image_version,header_optional_minor_image_version,header_optional_major_linker_version,...,section_entry_.upx1,section_entry_.vixen1,section_entry_.vmp1,section_entry_.vmp2,section_entry_.zdata,section_entry___OTHER__,section_entry_eintim,section_entry_init,section_entry_upx0,section_entry_upx1
0,1.726320e+09,6.0,0.0,240.0,0.0,2.0,1.0,0.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0
1,1.724938e+09,6.0,0.0,240.0,0.0,2.0,1.0,0.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0
2,1.699507e+09,5.0,0.0,240.0,0.0,3.0,1.0,6.0,1.0,9.0,...,0,0,0,0,0,0,0,0,0,0
3,1.710213e+09,6.0,0.0,240.0,0.0,3.0,1.0,0.0,0.0,12.0,...,0,0,0,0,0,0,0,0,0,0
4,1.590101e+09,6.0,0.0,240.0,0.0,3.0,1.0,10.0,0.0,14.0,...,0,0,0,0,0,0,0,0,0,0



FEATURE INDEX:
[0, 1, 2, 3, 4]

HASH ANALYSIS INDEX:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


STEP 13 — Verify the original extraction order
Instead of guessing or re-extracting immediately, inspect the previous notebook's extraction logic.

Run this cell in 04_dataset_1_ml_preparation.ipynb:

In [18]:

# STEP 13: INSPECT FEATURE EXTRACTION SOURCE CODE

from pathlib import Path
import json

project_root = Path.cwd().parent
notebook_path = project_root / "notebook" / "03_dataset_1_feature_extraction.ipynb"

print("NOTEBOOK FOUND:", notebook_path.exists())

if notebook_path.exists():
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    keywords = [
        "iterdir", "glob(", "sorted(",
        "sha256", "feature_rows", "append(",
        "to_csv", "extract", "exception"
    ]

    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        if any(k in source.lower() for k in keywords):
            print(f"\n{'='*65}")
            print("CELL:", i)
            print(source[:3500])


NOTEBOOK FOUND: True

CELL: 19

from collections import Counter

profile_results = []

for col in nested_columns:

    empty_count = 0
    non_empty_count = 0

    element_types = Counter()
    dictionary_keys = Counter()

    for value in df_structured[col]:

        if isinstance(value, list):

            if len(value) == 0:
                empty_count += 1
            else:
                non_empty_count += 1

            for item in value:

                element_types[type(item).__name__] += 1

                if isinstance(item, dict):
                    dictionary_keys.update(item.keys())

        elif isinstance(value, dict):

            non_empty_count += 1
            element_types["dict"] += 1
            dictionary_keys.update(value.keys())

    profile_results.append({
        "column": col,
        "empty": empty_count,
        "non_empty": non_empty_count,
        "element_types": dict(element_types),
        "dictionary_keys": list(dictionary_keys.keys())
    })

p

STEP 14 — Locate PE feature extraction and row ordering

In [19]:

import json
from pathlib import Path

nb_path = Path.cwd().parent / "notebook" / "03_dataset_1_feature_extraction.ipynb"

with open(nb_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "df_structured",
    "feature_rows",
    "feature_records",
    "sha256",
    "iterdir",
    "glob(",
    "os.listdir",
    "for file",
    "to_csv",
    "exception_records",
    "append("
]

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])
    matched = [k for k in keywords if k in source.lower()]

    if matched:
        print(f"\n{'='*60}")
        print(f"CELL {i} | MATCHES: {matched}")
        
        for line_no, line in enumerate(source.splitlines(), 1):
            if any(k in line.lower() for k in keywords):
                print(f"{line_no}: {line[:250]}")



CELL 8 | MATCHES: ['df_structured']
6:     df_structured = pickle.load(file)
9: print("Shape:", df_structured.shape)
11:       round(df_structured.memory_usage(deep=True).sum() / (1024**2), 2))
13: display(df_structured.head(3))

CELL 10 | MATCHES: ['df_structured']
29:     if col in df_structured.columns
36:     types = df_structured[col].dropna().map(

CELL 12 | MATCHES: ['df_structured']
5:     sample = df_structured[col].dropna()

CELL 14 | MATCHES: ['df_structured']
11:     for idx, value in df_structured[col].items():

CELL 16 | MATCHES: ['df_structured']
17:     for value in df_structured[col]:

CELL 19 | MATCHES: ['df_structured', 'append(']
14:     for value in df_structured[col]:
36:     profile_results.append({

CELL 22 | MATCHES: ['df_structured']
26:     for value in df_structured[col]:

CELL 25 | MATCHES: ['df_structured']
49:             value for value in df_structured[col]

CELL 28 | MATCHES: ['df_structured']
9: df_features = pd.DataFrame(index=df_structured.index)
1

STEP 15 — Inspect label preservation and feature extraction

In [20]:

import json
from pathlib import Path

nb_path = Path.cwd().parent / "notebook" / "03_dataset_1_feature_extraction.ipynb"

with open(nb_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

for cell_num in [28, 34, 56, 59, 71]:

    cell = nb["cells"][cell_num]

    print("\n" + "=" * 80)
    print(f"CELL {cell_num}")
    print("=" * 80)

    print("".join(cell["source"]))



CELL 28

# ============================================
# STEP 10: STRING AND GENERAL FEATURE EXTRACTION
# ============================================

# Create a separate feature DataFrame.
# Original structured dataset remains unchanged.

df_features = pd.DataFrame(index=df_structured.index)

# Preserve original target label.
df_features["label"] = df_structured["label"].copy()


# --------------------------------------------
# 1. Extract STRING features
# --------------------------------------------

def safe_dict(value):
    return value if isinstance(value, dict) else {}


string_data = df_structured["strings"].apply(safe_dict)

# Basic string statistics
string_basic_features = [
    "numstrings",
    "avlength",
    "printables",
    "entropy"
]

for feature in string_basic_features:

    df_features[f"strings_{feature}"] = (
        string_data.apply(lambda x: x.get(feature))
    )


# Expand string_counts dictionary
string_count_data = string_data.apply(
    lambda x: safe_di

In [22]:


import numpy as np
import pandas as pd

# Confirmed exception positions from Step 21
exception_positions = [1170, 9254]

# Create mask for the 9,998 retained records
keep_mask = np.ones(len(df_structured), dtype=bool)
keep_mask[exception_positions] = False

# Extract labels using exactly the same row exclusions
aligned_labels = df_structured.loc[keep_mask, "label"].reset_index(drop=True)

# Reset feature dataset index
df_features = df_features.reset_index(drop=True)

# Verify alignment
print("Original records:", len(df_structured))
print("Excluded records:", len(exception_positions))
print("Retained labels:", len(aligned_labels))
print("Feature records:", len(df_features))

assert len(aligned_labels) == len(df_features), \
    "Row alignment failed!"

# Attach target
df_features["label"] = aligned_labels.to_numpy()

# Final validation
print("\nFINAL DATASET SHAPE:", df_features.shape)

print("\nLABEL DISTRIBUTION:")
print(df_features["label"].value_counts())

print("\nMISSING LABELS:", df_features["label"].isna().sum())

print("\nLABEL DATA TYPE:", df_features["label"].dtype)

print("\nLABEL ATTACHMENT SUCCESSFUL")


Original records: 10000
Excluded records: 2
Retained labels: 9998
Feature records: 9998

FINAL DATASET SHAPE: (9998, 653)

LABEL DISTRIBUTION:
label
1    5000
0    4998
Name: count, dtype: int64

MISSING LABELS: 0

LABEL DATA TYPE: int64

LABEL ATTACHMENT SUCCESSFUL


/var/folders/25/kpc_zts11rs285txdrf5h0sh0000gn/T/ipykernel_18379/1255738520.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_features["label"] = aligned_labels.to_numpy()


Next: Step 13 — Identifier and data leakage removal

In [23]:

# STEP 13: IDENTIFIER AND DATA LEAKAGE INSPECTION

import re
import numpy as np
import pandas as pd

# Confirm target
assert "label" in df_features.columns

# Potential identifier/leakage columns
patterns = [
    "sha256", "sha1", "md5",
    "hash", "filename", "file_name",
    "filepath", "file_path",
    "sample_id", "record_id",
    "tlsh"
]

suspicious_columns = [
    col for col in df_features.columns
    if any(pattern in col.lower() for pattern in patterns)
]

print("DATASET SHAPE:", df_features.shape)
print("\nTARGET DISTRIBUTION:")
print(df_features["label"].value_counts())

print("\nPOTENTIAL IDENTIFIER / LEAKAGE COLUMNS:")
print(suspicious_columns)

print("\nNON-NUMERIC FEATURES:")
print(df_features.select_dtypes(
    exclude=np.number
).columns.tolist())

# Infinite values
numeric_df = df_features.select_dtypes(include=np.number)

infinite_counts = np.isinf(numeric_df).sum()
print("\nINFINITE VALUES:", infinite_counts.sum())

# Constant features excluding target
constant_columns = [
    col for col in df_features.columns
    if col != "label" and df_features[col].nunique(dropna=False) <= 1
]

print("\nCONSTANT FEATURES:", len(constant_columns))
print(constant_columns)

# Duplicate columns excluding target
feature_df = df_features.drop(columns=["label"])

duplicate_columns = feature_df.columns[
    feature_df.T.duplicated()
].tolist()

print("\nDUPLICATE FEATURES:", len(duplicate_columns))
print(duplicate_columns[:30])

print("\nLEAKAGE INSPECTION COMPLETED")


DATASET SHAPE: (9998, 653)

TARGET DISTRIBUTION:
label
1    5000
0    4998
Name: count, dtype: int64

POTENTIAL IDENTIFIER / LEAKAGE COLUMNS:
[]

NON-NUMERIC FEATURES:
[]

INFINITE VALUES: 0

CONSTANT FEATURES: 0
[]

DUPLICATE FEATURES: 15
['header_optional_subsystem_IMAGE_SUBSYSTEM_EFI_RUNTIME_DRIVER', 'header_optional_subsystem_IMAGE_SUBSYSTEM_WINDOWS_BOOT_APPLICATION', 'section_entry_.9k>', 'section_entry_.aspr2', 'section_entry_.cod2', 'section_entry_.dyl1', 'section_entry_.hexpthk', 'section_entry_.map2', 'section_entry_.mpress2', 'section_entry_.of2', 'section_entry_.rbdata', 'section_entry_.sb2', 'section_entry_.text2', 'section_entry_.vixen1', 'section_entry_.zdata']

LEAKAGE INSPECTION COMPLETED


Step 14 — Remove duplicates and prepare train/test datasets

In [24]:

# STEP 14: FINAL FEATURE CLEANING AND TRAIN/TEST SPLIT

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# 1. Identify duplicate feature columns (exclude target)
X_temp = df_features.drop(columns=["label"])

duplicate_columns = X_temp.columns[
    X_temp.T.duplicated()
].tolist()

# 2. Remove duplicate columns
df_ml = df_features.drop(columns=duplicate_columns).copy()

# 3. Separate features and target
X = df_ml.drop(columns=["label"])
y = df_ml["label"].astype("int64")

# 4. Train/test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Create output directory
output_dir = Path("processed_dataset_1")
output_dir.mkdir(parents=True, exist_ok=True)

# 6. Save datasets
X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

df_ml.to_csv(output_dir / "dataset1_ml_ready.csv", index=False)

# 7. Final report
print("=" * 55)
print("FINAL ML DATASET PREPARATION")
print("=" * 55)

print("Original shape:", df_features.shape)
print("Duplicate features removed:", len(duplicate_columns))
print("Final shape:", df_ml.shape)

print("\nTRAINING SET:", X_train.shape)
print("TEST SET:", X_test.shape)

print("\nTRAIN LABEL DISTRIBUTION:")
print(y_train.value_counts())

print("\nTEST LABEL DISTRIBUTION:")
print(y_test.value_counts())

print("\nMISSING VALUES:", df_ml.isna().sum().sum())

print("\nSAVED FILES:")
for file in sorted(output_dir.iterdir()):
    if file.is_file():
        print("-", file.name)

print("\nML PREPARATION COMPLETED")


FINAL ML DATASET PREPARATION
Original shape: (9998, 653)
Duplicate features removed: 15
Final shape: (9998, 638)

TRAINING SET: (7998, 637)
TEST SET: (2000, 637)

TRAIN LABEL DISTRIBUTION:
label
1    4000
0    3998
Name: count, dtype: int64

TEST LABEL DISTRIBUTION:
label
0    1000
1    1000
Name: count, dtype: int64

MISSING VALUES: 0

SAVED FILES:
- X_test.csv
- X_train.csv
- categorical_mappings.json
- dataset1_ml_ready.csv
- dataset1_pe_features.csv
- removed_duplicate_features.csv
- y_test.csv
- y_train.csv

ML PREPARATION COMPLETED


Next: Step 15 — Feature scaling and preprocessing assessment
Before training, we should inspect feature ranges and distributions. PE features can have very different scales, and some models are sensitive to that.

Run this cell:

In [25]:

# STEP 15: FEATURE SCALING ASSESSMENT

import numpy as np
import pandas as pd

# Inspect training feature statistics
feature_stats = X_train.describe().T

feature_stats["range"] = (
    feature_stats["max"] - feature_stats["min"]
)

feature_stats["zero_variance"] = (
    feature_stats["std"] == 0
)

print("FEATURE STATISTICS:")
display(feature_stats.sort_values(
    "range", ascending=False
).head(20))

print("\nFEATURE SCALE SUMMARY:")
print("Minimum feature value:", X_train.min().min())
print("Maximum feature value:", X_train.max().max())
print("Features requiring investigation:",
      (feature_stats["range"] > 1000).sum())

print("\nDATA TYPES:")
print(X_train.dtypes.value_counts())

print("\nSCALING ASSESSMENT COMPLETED")


FEATURE STATISTICS:


,count,mean,std,min,25%,50%,75%,max,range,zero_variance
header_optional_image_base,7998.0,2.883558e+15,2.126085e+17,0.0,4.299162e+09,5.368709e+09,6.442451e+09,1.844674e+19,1.844674e+19,False
header_optional_sizeof_stack_reserve,7998.0,2.209830e+09,1.967118e+11,0.0,1.048576e+06,1.048576e+06,2.097152e+06,1.759219e+13,1.759219e+13,False
header_optional_sizeof_stack_commit,7998.0,2.199583e+09,1.967112e+11,0.0,4.096000e+03,4.096000e+03,4.096000e+03,1.759219e+13,1.759219e+13,False
header_optional_sizeof_heap_reserve,7998.0,9.650612e+06,7.683914e+08,0.0,1.048576e+06,1.048576e+06,1.048576e+06,6.871948e+10,6.871948e+10,False
header_coff_timestamp,7998.0,1.316862e+09,8.519629e+08,0.0,3.570122e+07,1.717659e+09,1.725891e+09,4.289678e+09,4.289678e+09,False
header_optional_checksum,7998.0,5.679820e+06,9.603580e+07,0.0,0.000000e+00,1.574380e+05,7.784882e+05,4.286776e+09,4.286776e+09,False
header_coff_pointer_to_symbol_table,7998.0,1.045472e+06,5.071642e+07,0.0,0.000000e+00,0.000000e+00,7.283200e+04,3.203063e+09,3.203063e+09,False
header_optional_sizeof_image,7998.0,3.394394e+06,2.401696e+07,6656.0,1.433600e+05,3.112960e+05,1.437696e+06,1.667486e+09,1.667479e+09,False
section_vsize_sum,7998.0,3.367463e+06,2.401513e+07,0.0,1.151370e+05,2.903140e+05,1.415184e+06,1.667460e+09,1.667460e+09,False
header_optional_sizeof_initialized_data,7998.0,1.543653e+06,2.365949e+07,0.0,6.451200e+04,1.167360e+05,3.168000e+05,1.666208e+09,1.666208e+09,False



FEATURE SCALE SUMMARY:
Minimum feature value: 0.0
Maximum feature value: 1.8446736926867194e+19
Features requiring investigation: 54

DATA TYPES:
float64    605
int64       32
Name: count, dtype: int64

SCALING ASSESSMENT COMPLETED


Next: Step 16 — Save and verify ML-ready datasets
Before model training, run this cell.

In [26]:

# STEP 16: FINAL ML DATASET VALIDATION AND EXPORT

from pathlib import Path
import pandas as pd
import numpy as np

output_dir = Path("../data/processed/ml_dataset_1")
output_dir.mkdir(parents=True, exist_ok=True)

# Verify target separation
assert "label" not in X_train.columns
assert "label" not in X_test.columns

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert X_train.columns.equals(X_test.columns)

assert not X_train.isna().any().any()
assert not X_test.isna().any().any()

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_test.to_numpy()).all()

# Save datasets
X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

# Save feature names
pd.Series(X_train.columns).to_csv(
    output_dir / "feature_names.csv",
    index=False,
    header=["feature"]
)

print("ML DATASET VALIDATION PASSED")
print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Target distribution:")
print(y_train.value_counts())
print("Output directory:", output_dir.resolve())


ML DATASET VALIDATION PASSED
Training: (7998, 637)
Testing: (2000, 637)
Target distribution:
label
1    4000
0    3998
Name: count, dtype: int64
Output directory: /Users/gouravsingh/Desktop/Ai-Ml-integrated-SOC-Platform/data/processed/ml_dataset_1
